# Inference Priority Decorator

The `inference_priority` decorator sets request priority for **all LLM calls within its scope**. It works with any LangChain `BaseChatModel` that declares a `priority` Pydantic field — including `ChatNVIDIADynamo`.

> **Priority convention:** lower number = higher priority. `priority=1` is the most urgent.

In agentic workflows, not all LLM calls are equally urgent. A 5-token intent classifier is on the critical path — the user is waiting — while a 500-token background research call has slack. `inference_priority` lets you express this with a single decorator instead of managing separate LLM instances or threading kwargs through every call.

**Key features:**
- Works as both a **decorator** (`@inference_priority(priority=10)`) and a **context manager** (`with inference_priority(priority=10):`)
- **Precedence**: decorator context > instance defaults
- **Nesting**: inner scopes fully replace outer scopes
- **Async-safe**: uses Python `contextvars`, so concurrent async tasks are isolated

## Development Setup

**NOTE** This section is only necessary if you want to run the notebook locally from source. If you just want to use the package, skip to the next section.

If you are working from the `langchain-nvidia` repository, this project uses [Poetry](https://python-poetry.org/) to manage dependencies. Run the following from the `libs/ai-endpoints` directory:

```bash
cd libs/ai-endpoints
pip install poetry                         # if not already installed
poetry config virtualenvs.in-project true --local
poetry install --with test
```

This creates a `.venv` inside `libs/ai-endpoints/`. Then install `ipykernel` directly via the venv's pip and register the Jupyter kernel:

```bash
.venv/bin/pip install ipykernel
.venv/bin/python -m ipykernel install --user --name langchain-nvidia --display-name "langchain-nvidia"
```

After this, reload your editor window and select the **langchain-nvidia** kernel in the notebook kernel picker.

## Install the Package

In [ ]:
# Install from local source (inference_priority is not yet in a published release)
%pip install --upgrade --quiet ../../

## Prerequisites

This notebook targets a **local NIM deployment behind NVIDIA Dynamo**. You do not need an `NVIDIA_API_KEY` — the NIM is running on your infrastructure.

See the [Dynamo Quickstart Guide](https://docs.nvidia.com/dynamo/latest/getting-started/quickstart) to get started.

> **Important:** When deploying the model, ensure the Dynamo worker's `--context-length` is at least **2x** the `MAX_TOKENS` value configured below.

In [ ]:
NIM_BASE_URL = "http://localhost:8099/v1"
MODEL = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
MAX_TOKENS = 4096

## Basic Usage — Decorator

With `inference_priority`, you use **one LLM instance** and let the decorator control priority per-function.

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIADynamo, inference_priority

# Single LLM instance — priority is set by context, not by the instance
llm = ChatNVIDIADynamo(
    base_url=NIM_BASE_URL,
    model=MODEL,
    max_completion_tokens=MAX_TOKENS,
)


@inference_priority(priority=1)
def classify_intent(query: str) -> str:
    """Critical path — highest priority (1), short response."""
    return llm.invoke(query, osl=10).content


@inference_priority(priority=10)
def background_research(query: str) -> str:
    """Background work — low priority (10), long response."""
    return llm.invoke(query, osl=500).content


intent = classify_intent("My invoice has the wrong amount")
print(f"Intent: {intent}")

research = background_research("What are common billing dispute policies?")
print(f"Research: {research[:100]}...")

## Basic Usage — Context Manager

The same priority control works as a `with` block. This is handy when you want to set priority for a group of calls without defining a separate function.

In [ ]:
from langchain_nvidia_ai_endpoints import inference_priority

# High-priority block — user is waiting
with inference_priority(priority=1):
    intent = llm.invoke("My invoice has the wrong amount", osl=10).content
    print(f"Intent: {intent}")

# Low-priority block — background work
with inference_priority(priority=10):
    research = llm.invoke(
        "What are common billing dispute policies?", osl=500
    ).content
    print(f"Research: {research[:100]}...")

## More Examples

Here are more examples showing high-priority and low-priority functions side by side.

In [ ]:
@inference_priority(priority=1)
def classify_sentiment(text: str) -> str:
    """High-priority classification — short response."""
    return llm.invoke(
        f"Classify as positive or negative: '{text}'",
        osl=5,
    ).content


@inference_priority(priority=10)
def summarize_topic(topic: str) -> str:
    """Low-priority background work — long response."""
    return llm.invoke(
        f"Summarize the history of {topic}.",
        osl=500,
    ).content


print(f"Classification: {classify_sentiment('I love this product!')}")
print(f"Summary: {summarize_topic('customer loyalty programs')[:100]}...")

## Precedence

> **Reminder:** lower number = higher priority. `priority=1` is the most urgent.

The priority hierarchy is:

| Level | Example | Wins? |
|-------|---------|-------|
| Decorator / context manager | `@inference_priority(priority=1)` | When set |
| Instance default | `ChatNVIDIADynamo(priority=10)` | When no decorator |

This means you can set a low-priority default on the instance and override it per-function with the decorator.

In [ ]:
llm_with_default = ChatNVIDIADynamo(
    base_url=NIM_BASE_URL,
    model=MODEL,
    max_completion_tokens=MAX_TOKENS,
    priority=10,  # instance default: low priority
)


@inference_priority(priority=1)
def urgent_classification(query: str) -> str:
    """Decorator overrides the instance default of priority=10."""
    return llm_with_default.invoke(query, osl=10).content


# Without the decorator, the instance default (priority=10) is used
result = llm_with_default.invoke("Background task", osl=500)
print(f"Default priority result: {result.content[:80]}...")

# With the decorator, priority=1 overrides the instance default
print(f"Decorator priority result: {urgent_classification('Classify: urgent or not')}")

## Priority Scheduling Under Sustained Load

This test demonstrates that Dynamo's priority scheduler provides **fine-grained, monotonic priority ordering** when the GPU is under sustained contention.

### Architecture

1. **100 background workers** loop continuously at `priority=10`, keeping the GPU queue permanently saturated
2. After a 5-second ramp-up, **95 test requests** are injected simultaneously across 5 priority tiers
3. **All requests use identical parameters** (`max_tokens=2048`, `osl=2048`) — **only priority varies**
4. Each test request uses `@inference_priority` to set its tier's priority

```
Background: 100 workers looping at priority=10 (sustained GPU saturation)
                              │
After ramp-up, inject test:   │
  ├─ CRITICAL   (priority=1)  │   5 requests
  ├─ HIGH       (priority=3)  │  10 requests
  ├─ MEDIUM     (priority=5)  │  20 requests
  ├─ LOW        (priority=7)  │  40 requests
  └─ BACKGROUND (priority=10) │  20 requests
                              │         ──────────
                              │         95 test requests
```

If priority scheduling works, higher-priority tiers should complete faster — even though all requests generate the same number of tokens.

> **Why sustained load matters:** Without background pressure, the GPU can batch-process all requests simultaneously via continuous batching. Every request gets a token per forward pass, so they all finish at roughly the same time regardless of priority. Sustained load forces genuine queue contention, making priority scheduling observable.

In [ ]:
import asyncio
import statistics
import time
from dataclasses import dataclass
from typing import Any

import aiohttp
from langchain_nvidia_ai_endpoints import ChatNVIDIADynamo, inference_priority
from queries import QUERIES

# ── Test configuration ─────────────────────────────────────────────────────────
# Fixed for ALL requests — only priority changes between tiers.
TEST_MAX_TOKENS = 2048
OSL = 2048
LATENCY_SENSITIVITY = 1.0
IAT = 250

# Background load
BACKGROUND_WORKERS = 100
BACKGROUND_PRIORITY = 10
RAMP_UP_SECONDS = 5

# Test tiers — 95 requests injected per run
TEST_TIERS: list[dict[str, Any]] = [
    {"priority": 1,  "label": "CRITICAL",    "count": 5},
    {"priority": 3,  "label": "HIGH",        "count": 10},
    {"priority": 5,  "label": "MEDIUM",      "count": 20},
    {"priority": 7,  "label": "LOW",         "count": 40},
    {"priority": 10, "label": "BACKGROUND",  "count": 20},
]

# ── Single LLM instance — priority is set by decorator, not by the instance ───
test_llm = ChatNVIDIADynamo(
    base_url=NIM_BASE_URL,
    model=MODEL,
    max_completion_tokens=TEST_MAX_TOKENS,
)

# Override the default aiohttp session factory to allow longer request times.
# Under sustained load, low-priority requests can take 10+ minutes — well beyond
# aiohttp's default 300-second timeout.
_orig_ssl = test_llm._client._build_ssl_context()
test_llm._client.get_async_session_fn = lambda: aiohttp.ClientSession(
    connector=aiohttp.TCPConnector(ssl=_orig_ssl),
    timeout=aiohttp.ClientTimeout(total=1800),  # 30 minutes
)


# ── Result data class ─────────────────────────────────────────────────────────
@dataclass
class Result:
    name: str
    start_time: float
    end_time: float
    tier_label: str


# ── Print configuration ───────────────────────────────────────────────────────
test_total = sum(t["count"] for t in TEST_TIERS)
tier_labels = [t["label"] for t in TEST_TIERS]

print(f"Configuration:")
print(f"  max_tokens={TEST_MAX_TOKENS}, osl={OSL}")
print(f"  Background workers: {BACKGROUND_WORKERS} at priority={BACKGROUND_PRIORITY}")
print(f"  Ramp-up: {RAMP_UP_SECONDS}s")
print(f"  Query pool: {len(QUERIES)} unique (system, user) pairs")
print(f"\n  Test tiers ({test_total} total requests):")
for t in TEST_TIERS:
    print(f"    {t['label']:12s}  priority={t['priority']:2d}  count={t['count']:3d}")

In [ ]:
# ── Background worker — loops at lowest priority until stopped ─────────────────

@inference_priority(priority=BACKGROUND_PRIORITY)
async def background_worker(worker_id: int, stop_event: asyncio.Event, counter: list[int]) -> None:
    """Loop: send a request, await response, repeat — until stop_event is set."""
    q_idx = worker_id % len(QUERIES)
    while not stop_event.is_set():
        q = QUERIES[q_idx % len(QUERIES)]
        q_idx += 1
        msgs = [("system", q["system"]), ("user", q["user"])]
        try:
            await test_llm.ainvoke(msgs, osl=OSL)
            counter[0] += 1
        except Exception:
            await asyncio.sleep(0.5)


# ── Timed test call (no decorator — called from within decorated wrappers) ─────

async def _timed_call(
    name: str, t0: float, tier_label: str, msgs: list[tuple[str, str]],
) -> Result:
    start = time.perf_counter() - t0
    await test_llm.ainvoke(msgs, osl=OSL)
    end = time.perf_counter() - t0
    return Result(name=name, start_time=start, end_time=end, tier_label=tier_label)


# ── Decorated wrappers: one per priority tier ──────────────────────────────────

@inference_priority(priority=1)
async def critical_call(name, t0, tier_label, msgs):
    return await _timed_call(name, t0, tier_label, msgs)

@inference_priority(priority=3)
async def high_call(name, t0, tier_label, msgs):
    return await _timed_call(name, t0, tier_label, msgs)

@inference_priority(priority=5)
async def medium_call(name, t0, tier_label, msgs):
    return await _timed_call(name, t0, tier_label, msgs)

@inference_priority(priority=7)
async def low_call(name, t0, tier_label, msgs):
    return await _timed_call(name, t0, tier_label, msgs)

@inference_priority(priority=10)
async def background_call(name, t0, tier_label, msgs):
    return await _timed_call(name, t0, tier_label, msgs)

PRIORITY_WRAPPERS = {
    1: critical_call, 3: high_call, 5: medium_call,
    7: low_call, 10: background_call,
}


# ── Build test task list ──────────────────────────────────────────────────────

tasks_spec: list[tuple[str, str, int, list[tuple[str, str]]]] = []
query_idx = 0
for tier in TEST_TIERS:
    for i in range(tier["count"]):
        name = f"{tier['label'].lower()}_{i + 1:03d}"
        q = QUERIES[query_idx % len(QUERIES)]
        query_idx += 1
        msgs = [("system", q["system"]), ("user", q["user"])]
        tasks_spec.append((name, tier["label"], tier["priority"], msgs))


# ── Warmup ────────────────────────────────────────────────────────────────────

print("Warming up GPU with 20 concurrent calls...")
warmup_tasks = [test_llm.ainvoke("Warm up the GPU.", osl=500) for _ in range(20)]
await asyncio.gather(*warmup_tasks)
print("Warmup done.\n")


# ── Start background workers (run for entire test) ───────────────────────────

stop_event = asyncio.Event()
bg_counter: list[int] = [0]
bg_tasks = [
    asyncio.create_task(background_worker(i, stop_event, bg_counter))
    for i in range(BACKGROUND_WORKERS)
]
print(f"Started {BACKGROUND_WORKERS} background workers (priority={BACKGROUND_PRIORITY})")
print(f"Ramping up for {RAMP_UP_SECONDS}s to fill queue...")
await asyncio.sleep(RAMP_UP_SECONDS)
print(f"Ramp-up done. Background requests completed so far: {bg_counter[0]}\n")


# ── Fire all test requests simultaneously ─────────────────────────────────────

print(f"Firing {test_total} test requests simultaneously...")
t0 = time.perf_counter()
coros = []
for name, tier_label, pri, msgs in tasks_spec:
    wrapper = PRIORITY_WRAPPERS[pri]
    coros.append(wrapper(name, t0, tier_label, msgs))

results = list(await asyncio.gather(*coros))
wall = max(r.end_time for r in results)
print(f"Done in {wall:.1f}s  (bg requests completed: {bg_counter[0]})\n")


# ── Stop background workers ──────────────────────────────────────────────────

print(f"Stopping background workers (total bg requests: {bg_counter[0]})...")
stop_event.set()
done, pending = await asyncio.wait(bg_tasks, timeout=30)
for t in pending:
    t.cancel()
if pending:
    await asyncio.wait(pending, timeout=5)
print("Background workers stopped.\n")


# ── Per-tier statistics ──────────────────────────────────────────────────────

print("=" * 70)
print("  RESULTS — Priority Scheduling Under Sustained Load")
print("=" * 70)
print()
print(f"  {'Tier':12s}  {'Priority':>8s}  {'Median':>8s}  {'Mean':>8s}  "
      f"{'Min':>8s}  {'Max':>8s}  {'n':>3s}")
print(f"  {'─' * 12}  {'─' * 8}  {'─' * 8}  {'─' * 8}  "
      f"{'─' * 8}  {'─' * 8}  {'─' * 3}")

final = {}
for tier in TEST_TIERS:
    lb = tier["label"]
    times = [r.end_time for r in results if r.tier_label == lb]
    med = statistics.median(times)
    final[lb] = med
    print(f"  {lb:12s}  {tier['priority']:>8d}  {med:>7.1f}s  "
          f"{statistics.mean(times):>7.1f}s  {min(times):>7.1f}s  "
          f"{max(times):>7.1f}s  {len(times):>3d}")

bg_time = final["BACKGROUND"]
print()
print(f"  Relative to BACKGROUND ({bg_time:.1f}s):")
for tier in TEST_TIERS:
    lb = tier["label"]
    if lb == "BACKGROUND":
        continue
    delta = final[lb] - bg_time
    print(f"    {lb:12s}: {delta:>+7.1f}s")

print()
print(f"  Background load: {BACKGROUND_WORKERS} workers, "
      f"{bg_counter[0]} total requests during test")

### Execution Timeline

The Gantt chart below shows all 95 test requests grouped by priority tier and color-coded. Under sustained background load (100 workers at priority=10), the priority scheduler creates a clear **staircase pattern** — higher-priority requests complete first, even though all requests use the same `max_tokens` and `osl`.

The dashed vertical lines mark when each tier's last request completes.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Color scheme: warm → cool as priority increases (lower = more urgent) ──────
TIER_COLORS = {
    "CRITICAL":   "#d32f2f",   # red
    "HIGH":       "#f57c00",   # orange
    "MEDIUM":     "#ffa000",   # amber
    "LOW":        "#00897b",   # teal
    "BACKGROUND": "#1565c0",   # blue
}

TIER_ORDER = {"CRITICAL": 0, "HIGH": 1, "MEDIUM": 2, "LOW": 3, "BACKGROUND": 4}


def make_gantt(ax, results: list[Result], title: str) -> None:
    """Draw a horizontal bar chart grouped by priority tier."""
    sorted_results = sorted(
        results,
        key=lambda r: (TIER_ORDER.get(r.tier_label, 99), r.end_time),
    )

    x_max = max(r.end_time for r in results) * 1.08
    n = len(sorted_results)
    y_labels = []

    for i, r in enumerate(reversed(sorted_results)):
        color = TIER_COLORS.get(r.tier_label, "#333333")
        duration = r.end_time - r.start_time
        ax.barh(i, duration, left=r.start_time, height=0.7,
                color=color, edgecolor="white", linewidth=0.3, alpha=0.85)
        y_labels.append(r.name)

    ax.set_yticks(range(n))
    ax.set_yticklabels(y_labels, fontsize=4, fontfamily="monospace")
    ax.set_xlim(0, x_max)
    ax.set_xlabel("Wall-clock time (seconds)", fontsize=10)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.grid(axis="x", linestyle="--", alpha=0.3)
    ax.set_axisbelow(True)

    # Dashed vertical lines at each tier's completion time
    for lb in TIER_ORDER:
        tier_results = [r for r in results if r.tier_label == lb]
        if tier_results:
            tier_done = max(r.end_time for r in tier_results)
            ax.axvline(tier_done, color=TIER_COLORS[lb], linestyle="--",
                       linewidth=1.5, alpha=0.6)
            ax.text(tier_done + x_max * 0.003, n - 0.3,
                    f"{lb} done: {tier_done:.1f}s",
                    fontsize=7, color=TIER_COLORS[lb], va="top",
                    fontweight="bold", rotation=90)


fig, ax = plt.subplots(1, 1, figsize=(14, 20))

make_gantt(ax, results,
           f"95 Test Requests Under Sustained Load — "
           f"@inference_priority (1, 3, 5, 7, 10)")

# Legend
legend_handles = [
    mpatches.Patch(color=TIER_COLORS[lb], label=f"{lb} (priority={t['priority']})")
    for t in TEST_TIERS
    for lb in [t["label"]]
]
ax.legend(handles=legend_handles, loc="lower right", fontsize=9)

plt.tight_layout()
plt.show()

## Before & After Comparison

### Before: Priority Baked Into Instances

Without `inference_priority`, **every priority tier needs its own `ChatNVIDIADynamo` instance** — even when all other parameters are identical. Priority is hidden inside infrastructure config:

```python
# Five instances — identical output config, only priority differs
llm_critical = ChatNVIDIADynamo(base_url=URL, model=MODEL, max_completion_tokens=2048, priority=1)
llm_high     = ChatNVIDIADynamo(base_url=URL, model=MODEL, max_completion_tokens=2048, priority=3)
llm_medium   = ChatNVIDIADynamo(base_url=URL, model=MODEL, max_completion_tokens=2048, priority=5)
llm_low      = ChatNVIDIADynamo(base_url=URL, model=MODEL, max_completion_tokens=2048, priority=7)
llm_bg       = ChatNVIDIADynamo(base_url=URL, model=MODEL, max_completion_tokens=2048, priority=10)

# Must remember which instance to use for each call — priority is implicit
results = await asyncio.gather(
    llm_critical.ainvoke(query, osl=2048),   # priority=1
    llm_high.ainvoke(query, osl=2048),       # priority=3
    llm_medium.ainvoke(query, osl=2048),     # priority=5
    llm_low.ainvoke(query, osl=2048),        # priority=7
    llm_bg.ainvoke(query, osl=2048),         # priority=10
)
```

### After: Priority in Business Logic

With `inference_priority`, **one LLM instance** handles all requests and **priority lives in the business logic** — on the function that calls the LLM:

```python
# One instance — priority is NOT baked in
llm = ChatNVIDIADynamo(base_url=URL, model=MODEL, max_completion_tokens=2048)

@inference_priority(priority=1)
async def critical_call(query):
    return await llm.ainvoke(query, osl=2048)

@inference_priority(priority=3)
async def high_call(query):
    return await llm.ainvoke(query, osl=2048)

@inference_priority(priority=5)
async def medium_call(query):
    return await llm.ainvoke(query, osl=2048)

@inference_priority(priority=7)
async def low_call(query):
    return await llm.ainvoke(query, osl=2048)

@inference_priority(priority=10)
async def background_call(query):
    return await llm.ainvoke(query, osl=2048)

# Priority is visible on each call site
results = await asyncio.gather(
    critical_call(query),      # priority=1
    high_call(query),          # priority=3
    medium_call(query),        # priority=5
    low_call(query),           # priority=7
    background_call(query),    # priority=10
)
```

### Summary

<table>
  <thead>
    <tr>
      <th style="text-align:left">Aspect</th>
      <th style="text-align:center">Priority Baked Into Instances</th>
      <th style="text-align:center"><code>@inference_priority</code> Decorator</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>LLM instances needed</td>
      <td style="text-align:center">One per priority tier (5 identical instances)</td>
      <td style="text-align:center; background-color:#f0f7f0"><strong>One instance for all tiers</strong></td>
    </tr>
    <tr>
      <td>Priority lives in</td>
      <td style="text-align:center">Infrastructure (instance config)</td>
      <td style="text-align:center; background-color:#f0f7f0"><strong>Business logic (function decorator)</strong></td>
    </tr>
    <tr>
      <td>Adding a new priority tier</td>
      <td style="text-align:center">Create another instance</td>
      <td style="text-align:center; background-color:#f0f7f0"><strong>Add one decorated wrapper</strong></td>
    </tr>
    <tr>
      <td>Change priority</td>
      <td style="text-align:center">Rewire calls to a different instance</td>
      <td style="text-align:center; background-color:#f0f7f0"><strong>Change decorator argument</strong></td>
    </tr>
    <tr>
      <td>Nesting support</td>
      <td style="text-align:center">&mdash;</td>
      <td style="text-align:center; background-color:#f0f7f0"><strong>Yes</strong> (inner replaces outer)</td>
    </tr>
    <tr>
      <td>Async-safe</td>
      <td style="text-align:center">Yes (separate instances)</td>
      <td style="text-align:center; background-color:#f0f7f0"><strong>Yes</strong> (contextvars)</td>
    </tr>
    <tr>
      <td>Changes to existing clients</td>
      <td style="text-align:center">None</td>
      <td style="text-align:center">None</td>
    </tr>
  </tbody>
</table>

## Related Topics

- [ChatNVIDIADynamo Notebook](nvidia_ai_endpoints_dynamo.ipynb) — Dynamo-specific parameters (`osl`, `iat`, `latency_sensitivity`)
- [NVIDIA Dynamo](https://developer.nvidia.com/dynamo) — open-source inference framework
- [Dynamo Quickstart Guide](https://docs.nvidia.com/dynamo/latest/getting-started/quickstart)
- [ChatNVIDIA Documentation](nvidia_ai_endpoints.ipynb) — standard ChatNVIDIA usage
- [langchain-nvidia-ai-endpoints README](https://github.com/langchain-ai/langchain-nvidia/blob/main/libs/ai-endpoints/README.md)